In [58]:
# Installation Java
!apt-get install openjdk-11-jdk-headless -qq > /dev/null

# Téléchargement Spark
!wget -q https://archive.apache.org/dist/spark/spark-3.4.1/spark-3.4.1-bin-hadoop3.tgz

# Extraction Spark
!tar -xzf spark-3.4.1-bin-hadoop3.tgz

# Installer findspark
!pip install -q findspark



In [59]:
import os
import findspark

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.4.1-bin-hadoop3"

findspark.init()


In [60]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("...") \
    .config("spark.network.timeout", "8000s") \
    .config("spark.executor.heartbeatInterval", "120s") \
    .config("spark.sql.shuffle.partitions", "100") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()


In [61]:
from google.colab import files

uploaded = files.upload()


Saving train_dataset.csv to train_dataset (1).csv


In [62]:
import pandas as pd

df_train=pd.read_csv('train_dataset.csv')


In [63]:
# Étape 1 : Nettoyage des colonnes et filtrage des classes

# Nettoyage des noms de colonnes : suppression des espaces superflus
df_train.columns = df_train.columns.str.strip()

# Filtrage pour ne garder que les classes pertinentes : Benign, DoS, Web Attack (on néglige Infiltration et Heartbleed car échanrillon trop faible et peut donc être vu comme du bruit)
df_train = df_train[df_train['Label'].isin(['Benign', 'DoS', 'Web Attack'])]

# Réinitialiser les index
df_train.reset_index(drop=True, inplace=True)
# 3. Supprimer les colonnes en doublon comme 'Fwd Header Length.1'
df_train = df_train.loc[:, ~df_train.columns.str.contains(r"\.\d+$")]

print(df_train.duplicated().sum())

0


In [64]:
import numpy as np

# Identifier les colonnes pouvant contenir des valeurs infinies ou NaN
# On va d'abord forcer la conversion en float et chercher les valeurs problématiques

# Remplacer les chaînes 'Infinity', 'NaN' (en tant que texte) par np.nan si présentes
df_train.replace(['Infinity', 'NaN', 'inf', '-inf'], np.nan, inplace=True)

# Convertir toutes les colonnes (sauf 'Label') en float si possible
for col in df_train.columns:
    if col != 'Label':
        df_train[col] = pd.to_numeric(df_train[col], errors='coerce')

# Compter les valeurs NaN après nettoyage
nan_counts = df_train.isna().sum()
cols_with_nans = nan_counts[nan_counts > 0]

cols_with_nans

,0


In [65]:
# Supprimer les lignes contenant des NaN (ici uniquement dans 'Flow Bytes/s')
df_train.dropna(inplace=True)

# Vérification après suppression
final_shape = df_train.shape
remaining_nans = df_train.isna().sum().sum()  # Total de NaN restants

final_shape, remaining_nans


((9000, 78), np.int64(0))

In [66]:
from pyspark.sql.functions import col, when, expr
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier, LinearSVC, GBTClassifier, OneVsRest
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator


In [67]:
# 3. Conversion en DataFrame Spark
spark_df = spark.createDataFrame(df_train)

In [68]:

from pyspark.sql.functions import monotonically_increasing_id
# 4. Nettoyage des noms de colonnes
spark_df = spark_df.toDF(*[c.strip() for c in spark_df.columns])



# Ajouter une colonne id unique à chaque ligne
spark_df = spark_df.withColumn("id", monotonically_increasing_id())


# 5. Encodage du label déjà présent
indexer = StringIndexer(inputCol="Label", outputCol="label_index", handleInvalid="skip")
spark_df = indexer.fit(spark_df).transform(spark_df)


In [69]:
# 6. Forcer les colonnes numériques à être bien typées en float
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import isnan, isnull


numeric_cols = [f.name for f in spark_df.schema.fields if f.name not in ["Label", "label_index"]]
# 6bis. Supprimer les lignes contenant NaN ou Inf dans les colonnes numériques
for col_name in numeric_cols:
    spark_df = spark_df.filter(~isnan(col(col_name))) \
                       .filter(~isnull(col(col_name))) \
                       .filter(~(col(col_name) == float("inf"))) \
                       .filter(~(col(col_name) == float("-inf")))
for col_name in numeric_cols:
    spark_df = spark_df.withColumn(col_name, col(col_name).cast(DoubleType()))


# 7. Assemblage des features
assembler = VectorAssembler(inputCols=numeric_cols, outputCol="features_assembled")
scaler = StandardScaler(inputCol="features_assembled", outputCol="features")


In [70]:
# 8. Définir les modèles
rf = RandomForestClassifier(featuresCol="features", labelCol="label_index", predictionCol="rf_pred")
svm = OneVsRest(classifier=LinearSVC(featuresCol="features", labelCol="label_index"),
                labelCol="label_index", featuresCol="features", predictionCol="svm_pred")
gbt_base = GBTClassifier(featuresCol="features", labelCol="label_index")
gbt = OneVsRest(classifier=gbt_base, featuresCol="features", labelCol="label_index", predictionCol="gbt_pred")

In [71]:
spark_df.count()

8996

In [72]:
# 9. Pipelines pour chaque modèle
pipeline_rf = Pipeline(stages=[assembler, scaler, rf])
pipeline_svm = Pipeline(stages=[assembler, scaler, svm])
pipeline_gbt = Pipeline(stages=[assembler, scaler, gbt])

In [73]:
# 10. Entraîner les modèles
model_rf = pipeline_rf.fit(spark_df)
model_svm = pipeline_svm.fit(spark_df)
model_gbt = pipeline_gbt.fit(spark_df)

In [74]:
from google.colab import files

uploaded2 = files.upload()

Saving test_dataset.csv to test_dataset (1).csv


In [75]:
# Charger le dataset de test
df_test_pandas = pd.read_csv("test_dataset.csv")

# Nettoyage des colonnes dupliquées
df_test_pandas = df_test_pandas.loc[:, ~df_test_pandas.columns.str.contains(r"\.\d+$")]
print(df_test_pandas.duplicated().sum())


0


In [77]:
df_test_pandas[" Label"].value_counts()

,count
Label,
Benign,7400
DoS,2000
Web Attack,600


In [78]:
spark_df_test = spark.createDataFrame(df_test_pandas)
spark_df_test.count()

10000

In [79]:
spark_df_test = spark.createDataFrame(df_test_pandas)

# Nettoyage des noms de colonnes
spark_df_test = spark_df_test.toDF(*[c.strip() for c in spark_df_test.columns])

# Cast des colonnes numériques
numeric_cols = [c for c in spark_df_test.columns if c not in ["Label"]]
for col_name in numeric_cols:
    spark_df_test = spark_df_test.withColumn(col_name, col(col_name).cast(DoubleType()))

spark_df_test = spark_df_test.withColumn("id", monotonically_increasing_id())

In [81]:

spark_df_test = indexer.fit(spark_df_test).transform(spark_df_test)

In [82]:
spark_df_test.count()

10000

In [83]:
# Prédictions avec chaque modèle
pred_rf_test = model_rf.transform(spark_df_test).select("id", "label_index", "rf_pred")
pred_svm_test = model_svm.transform(spark_df_test).select("id", "label_index", "svm_pred")
pred_gbt_test = model_gbt.transform(spark_df_test).select("id", "label_index", "gbt_pred")

In [85]:
predictions_test = pred_rf_test.select("id", "label_index", "rf_pred") \
    .join(pred_svm_test.select("id", "svm_pred"), on="id") \
    .join(pred_gbt_test.select("id", "gbt_pred"), on="id")

In [87]:
predictions_test = predictions_test.withColumn(
    "final_prediction",
    expr("array_max(array(double(rf_pred), double(svm_pred), double(gbt_pred)))")
)

In [86]:
predictions_test.count()

10000

In [88]:
evaluator = MulticlassClassificationEvaluator(
    labelCol="label_index", predictionCol="final_prediction", metricName="f1"
    )
f1_score = evaluator.evaluate(predictions_test)
print(f"✅ F1-score with majority voting: {f1_score:.4f}")

✅ F1-score with majority voting: 0.6854


In [89]:
correct_preds_test = predictions_test.filter(col("final_prediction") == col("label_index")).count()
total_preds = predictions_test.count()
accuracy = correct_preds_test / total_preds * 100

print(f"✅ Accuracy with majority voting: {accuracy:.2f}%")


✅ Accuracy with majority voting: 65.60%
